# hai07 - ukur kecepatan latih di TPU (tahap B, jalur TPU gratis)
Input (Add Input): dataset **hai07-bench** (unggah ulang `kaggle/hai07-bench.zip`; dibuat oleh `python tools/package_bench_kit.py`, sekarang memuat `bench_train_xla.py`).
Settings: Accelerator = **TPU VM v5e-8** (atau v6e-8 bila itu yang tampil; catat mana yang Anda pilih). **Internet ON** hanya diperlukan bila `torch_xla` belum terpasang (sel 2 akan memberi tahu).
**Hemat kuota (20 jam per minggu):** sel 3 menjalankan uji asap kecil dulu dan **berhenti** bila gagal. Jangan jalankan sel 4 sebelum sel 3 berhasil. Perkiraan total ±30-45 menit (kompilasi XLA di langkah pertama tiap konfigurasi lambat).
Kode TPU ini belum pernah dijalankan penuh oleh pembuatnya. Percobaan pertama gagal di mode 8 proses (TPU Kaggle: 'Expected 8 worker addresses, got 1'); sekarang bawaannya mode SPMD (satu proses mengendalikan 8 chip). Kirim seluruh keluaran sel yang gagal.
Hasil: token/detik total 8 chip, MFU kasar, memori; dasar untuk memutuskan apakah 540M di TPU Kaggle realistis (kuota 20 jam per minggu).

In [ ]:
import glob, os, subprocess, sys, zipfile, json
WORK = "/kaggle/working/hai07-bench"; os.makedirs(WORK, exist_ok=True)
src_dir = next((os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/tools/bench_train_xla.py", recursive=True)), None)
if src_dir: subprocess.run(["cp", "-r", src_dir + "/.", WORK], check=True)
else:
    z = glob.glob("/kaggle/input/**/hai07-bench.zip", recursive=True); assert z, "tambahkan dataset hai07-bench sebagai Input (versi terbaru)"
    zipfile.ZipFile(z[0]).extractall(WORK)
os.chdir(WORK)
assert os.path.exists("tools/bench_train_xla.py"), "dataset hai07-bench lama: unggah ulang zip terbaru"
print("berkas siap di", WORK)


In [ ]:
import importlib, subprocess, sys
try:
    import torch, torch_xla
    print("torch", torch.__version__, "| torch_xla", torch_xla.__version__)
except ImportError as e:
    print("torch_xla belum terpasang:", e, "\nMencoba memasang (perlu Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "torch_xla[tpu]", "-f", "https://storage.googleapis.com/libtpu-releases/index.html"], check=False)
    import torch, torch_xla
    print("torch", torch.__version__, "| torch_xla", torch_xla.__version__)
import os
print("TPU_ACCELERATOR_TYPE =", os.environ.get("TPU_ACCELERATOR_TYPE"), "| PJRT_DEVICE =", os.environ.get("PJRT_DEVICE"))


In [ ]:
ACCEL = "v5e"   # ganti ke "v6e" bila akselerator yang Anda pilih v6e-8 (hanya untuk menghitung MFU)
# UJI ASAP: model sangat kecil, beberapa langkah. Berhenti bila gagal.
import subprocess, sys, json
p = subprocess.run([sys.executable, "-u", "tools/bench_train_xla.py", "--preset", "tiny", "--ctx", "64", "--batch", "2", "--steps", "3", "--warmup", "2", "--accelerator", ACCEL],
                   capture_output=True, text=True, timeout=900)
print(p.stdout[-3000:]); print(p.stderr[-4000:])
ok = any(l.startswith("{") and "token_per_detik" in l for l in p.stdout.splitlines())
assert ok, "UJI ASAP GAGAL: kirim keluaran di atas ke pengembang; jangan lanjut ke sel 4"
print("UJI ASAP BERHASIL")


In [ ]:
# (nama, argumen). batch = per chip. 1B diperkirakan tidak muat (parameter + optimizer 17 GB > memori satu chip; perlu FSDP yang belum ada).
RUNS = [
    ("100M ctx256 batch16/chip",                 ["--preset", "100m", "--ctx", "256",  "--batch", "16"]),
    ("100M ctx1024 batch8/chip",                 ["--preset", "100m", "--ctx", "1024", "--batch", "8"]),
    ("300M ctx1024 ckpt batch4/chip",            ["--preset", "300m", "--ctx", "1024", "--batch", "4", "--grad-checkpoint"]),
    ("540M ctx1024 ckpt batch2/chip",            ["--preset", "540m", "--ctx", "1024", "--batch", "2", "--grad-checkpoint"]),
    ("540M ctx1024 ckpt batch4/chip",            ["--preset", "540m", "--ctx", "1024", "--batch", "4", "--grad-checkpoint"]),
    ("1B ctx1024 ckpt batch1/chip (replika: diduga OOM)", ["--preset", "1b", "--ctx", "1024", "--batch", "1", "--grad-checkpoint"]),
    ("1B ctx1024 ckpt batch1/chip, PARAMETER DIPECAH (percobaan)", ["--preset", "1b", "--ctx", "1024", "--batch", "1", "--grad-checkpoint", "--shard-params"]),
]
OUT = "/kaggle/working/bench_xla.jsonl"
if os.path.exists(OUT): os.remove(OUT)
rows = []
for name, extra in RUNS:
    print("=" * 90); print(name, flush=True)
    try:
        p = subprocess.run([sys.executable, "-u", "tools/bench_train_xla.py", "--steps", "8", "--warmup", "3", "--accelerator", ACCEL, "--out", OUT] + extra,
                           capture_output=True, text=True, timeout=1800)
        last = [l for l in p.stdout.splitlines() if l.startswith("{")]
        if last:
            d = json.loads(last[-1]); d["nama"] = name; rows.append(d); print(last[-1])
        else:
            err = (p.stderr.strip().splitlines() or ["(tanpa keluaran)"])[-1][:200]
            rows.append({"nama": name, "galat": err}); print("GALAT:", err)
    except subprocess.TimeoutExpired:
        rows.append({"nama": name, "galat": "melewati batas waktu 30 menit"}); print("GALAT: batas waktu")


In [ ]:
cols = ["nama", "parameter_juta", "chip", "token_per_detik", "mfu_persen", "tflops_efektif_total", "memori_dipakai_gb", "jam_untuk_10_miliar_token", "galat"]
w = {c: max(len(c), *(len(str(r.get(c, "-"))) for r in rows)) for c in cols}
print("  ".join(c.ljust(w[c]) for c in cols)); print("  ".join("-" * w[c] for c in cols))
for r in rows: print("  ".join(str(r.get(c, "-")).ljust(w[c]) for c in cols))
print("\nPerangkat:", next((r.get("perangkat") for r in rows if r.get("perangkat")), "?"))
print("Kuota: 20 jam per minggu. 540M 10,8 miliar token butuh jam_untuk_10_miliar_token x 1,08 jam latih; bagi 20 untuk jumlah minggu.")
print("Salin tabel ini dan kirim ke pengembang. Berkas lengkap:", OUT)
